**Fake News Detection (LSTM)**

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv('/content/News.csv')

In [ ]:
df.head()

,Unnamed: 0,News Title,News Text,subject,News Date,Class
0,0.0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0.0
1,1.0,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0.0
2,2.0,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0.0
3,3.0,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0.0
4,4.0,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0.0


In [ ]:

df.isna().sum()

,0
Unnamed: 0,0
News Title,0
News Text,0
subject,0
News Date,0
Class,0


In [ ]:
#This is Used for Droping Non Values
df=df.dropna()

In [ ]:
X=df.drop('Class',axis=1)

In [ ]:
y=df['Class']

In [ ]:
X.shape

(44898, 5)

In [ ]:
y.shape

(44898,)

In [ ]:
import tensorflow as tf

In [ ]:
tf.__version__

'2.17.1'

In [ ]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [ ]:
# Tokenizer setup
MAX_WORDS = 10000  # Top 10,000 most common words
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df['News Text'])

In [ ]:
voc_size = len(tokenizer.word_index) + 1
MAX_LEN = 200

Now One Hot Representation

In [ ]:
messages=X.copy()

In [ ]:
messages['News Title'][1]

' Drunk Bragging Trump Staffer Started Russian Collusion Investigation'

In [ ]:
messages.reset_index(inplace=True)

In [ ]:
import nltk
import re
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    review = re.sub('[^a-zA-Z]', ' ', messages['News Title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

Streaming output truncated to the last 5000 lines.
39898
39899
39900
39901
39902
39903
39904
39905
39906
39907
39908
39909
39910
39911
39912
39913
39914
39915
39916
39917
39918
39919
39920
39921
39922
39923
39924
39925
39926
39927
39928
39929
39930
39931
39932
39933
39934
39935
39936
39937
39938
39939
39940
39941
39942
39943
39944
39945
39946
39947
39948
39949
39950
39951
39952
39953
39954
39955
39956
39957
39958
39959
39960
39961
39962
39963
39964
39965
39966
39967
39968
39969
39970
39971
39972
39973
39974
39975
39976
39977
39978
39979
39980
39981
39982
39983
39984
39985
39986
39987
39988
39989
39990
39991
39992
39993
39994
39995
39996
39997
39998
39999
40000
40001
40002
40003
40004
40005
40006
40007
40008
40009
40010
40011
40012
40013
40014
40015
40016
40017
40018
40019
40020
40021
40022
40023
40024
40025
40026
40027
40028
40029
40030
40031
40032
40033
40034
40035
40036
40037
40038
40039
40040
40041
40042
40043
40044
40045
40046
40047
40048
40049
40050
40051
40052
40053
40054
40055
4

In [ ]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[54654, 15901, 84785, 86917, 79562, 107082, 104085, 77518, 2806],
 [53939, 88912, 15901, 134017, 100417, 116355, 67994, 121569],
 [59557, 79207, 38770, 67379, 52323, 116416, 3257, 124870, 40490, 129246],
 [15901, 95869, 35567, 114201, 95818, 134571, 57133, 1699],
 [81279, 114373, 102023, 54654, 15901, 129141, 122582],
 [27314, 53704, 48482, 108596, 96570, 68430, 15362, 99661, 1699],
 [8668, 109080, 92141, 15901, 75362, 24243, 112186, 58480, 53249, 136416],
 [15901, 35532, 89157, 27314, 14593, 114882, 60356, 60559, 124371, 124409],
 [12718,
  133165,
  58480,
  7152,
  15901,
  121202,
  45871,
  27059,
  91702,
  38486,
  26627,
  110009,
  125170],
 [23951, 99196, 79562, 57928, 15901, 509, 14208, 133041, 23488, 22601, 118028],
 [67675, 47360, 22848, 65643, 61113, 99367, 129912, 61495],
 [23951, 97079, 73634, 40559, 43848, 4593, 48595, 45135, 7392, 11070, 64818],
 [129912, 58480, 15901, 102524, 5320, 45761, 131606, 59851],
 [23951, 125007, 1265, 121827, 76918, 46426, 15901, 104198, 54

**Embedding Representation**

In [ ]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[     0      0      0 ... 104085  77518   2806]
 [     0      0      0 ... 116355  67994 121569]
 [     0      0      0 ... 124870  40490 129246]
 ...
 [     0      0      0 ...  98252  67379  26248]
 [     0      0      0 ... 114373  50424  84518]
 [     0      0      0 ...  66304 116355 101189]]


In [ ]:
embedded_docs[0]

array([     0,      0,      0,      0,      0,      0,      0,      0,
            0,      0,      0,  54654,  15901,  84785,  86917,  79562,
       107082, 104085,  77518,   2806], dtype=int32)

In [ ]:
#Creating Model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
#Complile the model
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ (32, 20, 40)                │       5,504,520 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (32, 20, 40)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ (32, 100)                   │          56,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (32, 100)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (32, 1)                     │             101 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,561,021 (21.21 MB)

 Trainable params: 5,561,021 (21.21 MB)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
len(embedded_docs)

44898

In [ ]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [ ]:
X_final.shape,y_final.shape

((44898, 20), (44898,))

In [ ]:
sequences = tokenizer.texts_to_sequences(df['News Text'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

**Model Training**

In [ ]:
#Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 46s 92ms/step - accuracy: 0.8392 - loss: 0.3249 - val_accuracy: 0.9515 - val_loss: 0.1236
Epoch 2/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 40s 86ms/step - accuracy: 0.9689 - loss: 0.0845 - val_accuracy: 0.9533 - val_loss: 0.1295
Epoch 3/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 43s 91ms/step - accuracy: 0.9811 - loss: 0.0508 - val_accuracy: 0.9525 - val_loss: 0.1378
Epoch 4/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 42s 88ms/step - accuracy: 0.9903 - loss: 0.0290 - val_accuracy: 0.9513 - val_loss: 0.1663
Epoch 5/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 83s 90ms/step - accuracy: 0.9940 - loss: 0.0187 - val_accuracy: 0.9485 - val_loss: 0.1932
Epoch 6/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 83s 92ms/step - accuracy: 0.9961 - loss: 0.0132 - val_accuracy: 0.9483 - val_loss: 0.2298
Epoch 7/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 82s 92ms/step - accuracy: 0.9966 - loss: 0.0102 - val_accuracy: 0.9510 - val_loss: 0.2486
Epoch 8/10
471/471 ━━━━━━━━━━━━━━━━━━━━ 90s 109ms/step - accuracy: 0.9980 - loss: 0.0070 -

**Performance Metrices And Accuracy**

In [ ]:
y_pred=(model.predict(X_test) > 0.5).astype("int32")

464/464 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step


In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
confusion_matrix(y_test,y_pred)

array([[4987, 2791],
       [4862, 2177]])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.48349868394411827

In [ ]:
MAX_LEN = 200  # Maximum sequence length
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN)

In [ ]:
X = padded_sequences
y = df['Class']

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

464/464 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step


In [ ]:
# Generate predictions
y_pred = (model.predict(X_test) > 0.5).astype("int32")

# Check accuracy or other metrics
from sklearn.metrics import accuracy_score, classification_report

# Evaluate performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


464/464 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step
Accuracy: 0.48349868394411827
Classification Report:
               precision    recall  f1-score   support

         0.0       0.51      0.64      0.57      7778
         1.0       0.44      0.31      0.36      7039

    accuracy                           0.48     14817
   macro avg       0.47      0.48      0.46     14817
weighted avg       0.47      0.48      0.47     14817

